# Customer Churn Prediction
### CodSoft Machine Learning Internship — Task 3
---
**Dataset:** Bank Customer Churn Prediction  
**Source:** [Kaggle](https://www.kaggle.com/datasets/shantanudhakadd/bank-customer-churn-prediction)  
**Goal:** Predict which customers are likely to leave a bank using historical data.

**Algorithms compared:**
- Logistic Regression (interpretable baseline)
- Random Forest (ensemble method)
- Gradient Boosting (boosted trees — typically our strongest model)


## 0. Imports & Configuration

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics         import (
    classification_report, confusion_matrix, roc_curve,
    auc, accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score,
)

warnings.filterwarnings("ignore")

# -- plot style --
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.facecolor": "#FAFAFA",
    "axes.facecolor":   "#FAFAFA",
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right":False,
})

PALETTE = {
    "Logistic Regression": "#4E8FD4",
    "Random Forest":        "#27AE60",
    "Gradient Boosting":    "#E74C3C",
}

print("Setup complete ✓")


## 1. Load & Inspect the Data

In [ ]:
df = pd.read_csv("../data/Churn_Modelling.csv")
print(f"Shape: {df.shape}")
df.head()


In [ ]:
print("Data Types:")
print(df.dtypes)
print(f"\nMissing values: {df.isnull().sum().sum()}")
df.describe()


## 2. Exploratory Data Analysis (EDA)
Let's understand the data before touching any models.

In [ ]:
# -- Target variable distribution --
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df["Exited"].value_counts()
axes[0].bar(["Stayed", "Churned"], counts.values,
            color=["#4E8FD4", "#E74C3C"], width=0.5, edgecolor="white", linewidth=1.5)
axes[0].set_title("Churn Count", fontweight="bold")
axes[0].set_ylabel("Customers")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha="center", fontweight="bold")

axes[1].pie(counts.values, labels=["Stayed", "Churned"],
            autopct="%1.1f%%", colors=["#4E8FD4", "#E74C3C"],
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Churn Split", fontweight="bold")

fig.suptitle("Target Variable Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

churn_rate = df["Exited"].mean() * 100
print(f"Overall churn rate: {churn_rate:.2f}%")
print("The dataset is moderately imbalanced — we'll use stratified splits and AUC as our primary metric.")


In [ ]:
# -- Numerical feature distributions by churn status --
num_cols = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    stayed  = df[df["Exited"] == 0][col]
    churned = df[df["Exited"] == 1][col]
    ax.hist(stayed,  bins=30, alpha=0.65, color="#4E8FD4", label="Stayed",  density=True)
    ax.hist(churned, bins=30, alpha=0.65, color="#E74C3C", label="Churned", density=True)
    ax.set_title(col, fontweight="bold")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

axes[-1].axis("off")
fig.suptitle("Numerical Features: Stayed vs Churned", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# -- Churn rate by categorical features --
cat_cols = ["Geography", "Gender", "NumOfProducts", "HasCrCard", "IsActiveMember"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ax = axes[i]
    rates = df.groupby(col)["Exited"].mean().reset_index()
    rates.columns = [col, "ChurnRate"]
    sns.barplot(data=rates, x=col, y="ChurnRate", ax=ax, palette="Set2")
    ax.set_title(f"Churn Rate by {col}", fontweight="bold")
    ax.set_ylabel("Churn Rate")
    ax.set_ylim(0, 0.75)
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.2f}",
                    (p.get_x() + p.get_width() / 2, p.get_height() + 0.01),
                    ha="center", fontsize=9)

axes[-1].axis("off")
fig.suptitle("Churn Rate by Category", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# -- Correlation heatmap --
num_df = df.select_dtypes(include=[np.number]).drop(
    columns=["RowNumber", "CustomerId"], errors="ignore")

corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 3. Data Preprocessing

In [ ]:
df_clean = df.copy()

# Drop columns that are just identifiers — not useful for prediction
df_clean.drop(columns=["RowNumber", "CustomerId", "Surname"], inplace=True)

# Encode Gender: Male → 1, Female → 0
le = LabelEncoder()
df_clean["Gender"] = le.fit_transform(df_clean["Gender"])

# One-hot encode Geography
geo_dummies = pd.get_dummies(df_clean["Geography"], prefix="Geo", drop_first=False)
df_clean = pd.concat([df_clean.drop(columns=["Geography"]), geo_dummies], axis=1)

print("Columns after preprocessing:")
print(list(df_clean.columns))
df_clean.head()


In [ ]:
X = df_clean.drop(columns=["Exited"])
y = df_clean["Exited"]

feature_names = list(X.columns)

# Stratified split — preserves churn ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Standardise numerical features
scaler   = StandardScaler()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Features: {len(feature_names)}")
print(f"Churn rate in train: {y_train.mean()*100:.2f}%  | test: {y_test.mean()*100:.2f}%")


## 4. Model Training

We train three algorithms and compare their performance.  
Each model gets a quick hyperparameter search via `GridSearchCV` using `roc_auc` as the scoring metric — that's the right choice for imbalanced churn data.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Logistic Regression ────────────────────────────────────────────────────
print("Training Logistic Regression ...")
lr_grid = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=500),
    param_grid={"C": [0.01, 0.1, 1, 10], "class_weight": ["balanced", None]},
    scoring="roc_auc", cv=cv, n_jobs=-1,
)
lr_grid.fit(X_train, y_train)
best_lr  = lr_grid.best_estimator_
print(f"  Best CV AUC: {lr_grid.best_score_:.4f}  Params: {lr_grid.best_params_}")


In [ ]:
# ── Random Forest ──────────────────────────────────────────────────────────
print("Training Random Forest ...")
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={
        "n_estimators": [100, 200],
        "max_depth":    [6, 10, None],
        "class_weight": ["balanced", None],
    },
    scoring="roc_auc", cv=cv, n_jobs=-1,
)
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
print(f"  Best CV AUC: {rf_grid.best_score_:.4f}  Params: {rf_grid.best_params_}")


In [ ]:
# ── Gradient Boosting ──────────────────────────────────────────────────────
print("Training Gradient Boosting ...")
gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid={
        "n_estimators":  [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth":     [3, 5],
    },
    scoring="roc_auc", cv=cv, n_jobs=-1,
)
gb_grid.fit(X_train, y_train)
best_gb = gb_grid.best_estimator_
print(f"  Best CV AUC: {gb_grid.best_score_:.4f}  Params: {gb_grid.best_params_}")


In [ ]:
models = {
    "Logistic Regression": best_lr,
    "Random Forest":        best_rf,
    "Gradient Boosting":    best_gb,
}
print("All models trained ✓")


## 5. Model Evaluation

In [ ]:
summary = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"\n{'─'*55}")
    print(f"  {name}")
    print(f"{'─'*55}")
    print(classification_report(y_test, y_pred, target_names=["Stayed", "Churned"]))

    summary.append({
        "Model":     name,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall":    recall_score(y_test, y_pred),
        "F1":        f1_score(y_test, y_pred),
        "ROC-AUC":   roc_auc_score(y_test, y_prob),
    })


In [ ]:
# -- Confusion matrices --
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    color = PALETTE[name]
    sns.heatmap(cm, annot=True, fmt="d",
                cmap=sns.light_palette(color, as_cmap=True),
                xticklabels=["Stayed", "Churned"],
                yticklabels=["Stayed", "Churned"],
                ax=ax, linewidths=0.5, cbar=False)
    ax.set_title(name, fontweight="bold")
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")

fig.suptitle("Confusion Matrices", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# -- ROC Curves --
fig, ax = plt.subplots(figsize=(7, 5))

for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{name}  (AUC = {roc_auc:.3f})",
            color=PALETTE[name], linewidth=2.2)

ax.plot([0, 1], [0, 1], "--", color="#95a5a6", linewidth=1.2, label="Random guess")
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")
ax.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# -- Metrics comparison table and chart --
df_summary = pd.DataFrame(summary).set_index("Model")
print(df_summary.to_string(float_format=lambda x: f"{x:.4f}"))

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
x = np.arange(len(metrics))
w = 0.22

fig, ax = plt.subplots(figsize=(11, 5))
for i, (idx, row) in enumerate(df_summary.iterrows()):
    offset = (i - 1) * w
    bars = ax.bar(x + offset, row[metrics].values, w, label=idx,
                  color=PALETTE[idx], alpha=0.85, edgecolor="white")
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005, f"{h:.2f}",
                ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.12); ax.set_ylabel("Score")
ax.set_title("Model Comparison — All Metrics", fontsize=13, fontweight="bold")
ax.legend(); ax.grid(axis="y")
plt.tight_layout()
plt.show()


In [ ]:
# -- Feature Importance --
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, model) in zip(axes, models.items()):
    if hasattr(model, "feature_importances_"):
        imps = model.feature_importances_
    else:
        imps = np.abs(model.coef_[0])

    top = np.argsort(imps)[::-1][:12]
    vals = imps[top]
    labs = [feature_names[i] for i in top]

    ax.barh(range(12), vals[::-1], color=PALETTE[name], alpha=0.8, edgecolor="white")
    ax.set_yticks(range(12)); ax.set_yticklabels(labs[::-1], fontsize=8)
    ax.set_title(f"{name}\nTop Features", fontweight="bold")
    ax.set_xlabel("Importance")

fig.suptitle("Feature Importances by Model", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Final Summary & Key Insights

In [ ]:
best_model_name = df_summary["ROC-AUC"].idxmax()
best_auc = df_summary.loc[best_model_name, "ROC-AUC"]

print("═" * 60)
print("  RESULTS SUMMARY")
print("═" * 60)
print(df_summary[["ROC-AUC", "F1", "Recall"]].to_string(float_format=lambda x: f"{x:.4f}"))
print()
print(f"  🏆 Best model: {best_model_name}  (ROC-AUC = {best_auc:.4f})")
print()
print("  KEY FINDINGS:")
print("  • Age is the strongest predictor — older customers churn more")
print("  • Inactive members are roughly twice as likely to churn")
print("  • German customers churn at a noticeably higher rate")
print("  • Customers with only one or three/four products churn more")
print("  • Account balance has a bimodal distribution — zero-balance")
print("    customers behave differently from high-balance ones")
print("═" * 60)


## 7. Business Recommendations

Based on the model's outputs, the bank should focus retention efforts on:

| Segment | Risk | Action |
|---|---|---|
| Age 40–60 | High | Personalised loyalty offers |
| Inactive members | High | Re-engagement campaigns |
| German customers | High | Region-specific incentives |
| Single-product customers | Medium | Cross-sell campaigns |
| High balance, inactive | High | Dedicated relationship manager |

> **Recall over Precision** — In churn prediction, missing a churner is more costly than a false alarm. Prioritise models with higher Recall when deploying to production.
